In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# dpo pairs from the



## Pairs for DPO

In [ ]:
!pip -q install -U transformers peft bitsandbytes bert-score

import os, json, random, gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from bert_score import BERTScorer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -----------------------
# PATHS / CONFIG
# -----------------------
BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685_final_project"

REALIZER_TRAIN_PATH = os.path.join(BASE_PATH, "data/realizer_train.jsonl")
OUT_PATH = os.path.join(BASE_PATH, "data/dpo_pairs_realiser_train_100_v3_gpu.jsonl")

MODEL_BASE = "issai/LLama-3.1-KazLLM-1.0-8B"
SFT_REALISER_DIR = os.path.join(BASE_PATH, "models/realizer_kazllm")

SPLIT_TOKEN = "[INST] Write the fairy tale based on this plan.\n"
N_EXAMPLES = 100
K_CANDIDATES = 3
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------
# 0) dynamic generation length (increase > 250)
# -----------------------
def max_new_tokens_from_prompt(prompt: str) -> int:
    if "<LENGTH=very_short>" in prompt: return 220
    if "<LENGTH=short>" in prompt:      return 360
    if "<LENGTH=medium>" in prompt:     return 650
    if "<LENGTH=long>" in prompt:       return 1000
    return 500

# -----------------------
# 1) load prompt/reference pairs from realizer_train
# -----------------------
pairs = []
with open(REALIZER_TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        text = obj["text"]
        i = text.find(SPLIT_TOKEN)
        if i == -1:
            continue
        prompt_end = i + len(SPLIT_TOKEN)
        prompt = text[:prompt_end]
        ref = text[prompt_end:].lstrip("\n")
        if ref:
            pairs.append((prompt, ref))

random.shuffle(pairs)
pairs = pairs[:N_EXAMPLES]
print("Loaded prompt/reference pairs:", len(pairs))

# -----------------------
# 2) load policy (LLM) in 4bit on GPU
# -----------------------
gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tok = AutoTokenizer.from_pretrained(MODEL_BASE, token=True, use_fast=True)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=True,
    quantization_config=bnb,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
policy = PeftModel.from_pretrained(base, SFT_REALISER_DIR)
policy.eval()
policy.config.pad_token_id = tok.eos_token_id

@torch.inference_mode()
def generate_one(prompt: str) -> str:
    mnt = max_new_tokens_from_prompt(prompt)
    inputs = tok(prompt, return_tensors="pt").to(policy.device)
    input_len = inputs["input_ids"].shape[1]

    out = policy.generate(
        **inputs,
        do_sample=True,
        temperature=0.85,
        top_p=0.92,
        max_new_tokens=mnt,
        min_new_tokens=min(100, mnt),
        repetition_penalty=1.05,
        pad_token_id=tok.eos_token_id,
        eos_token_id=tok.eos_token_id,
    )[0]

    gen_ids = out[input_len:]  # строго новые токены
    return tok.decode(gen_ids, skip_special_tokens=True).strip()

# -----------------------
# 3) BERTScore on GPU (loads once)
# -----------------------
scorer = BERTScorer(
    model_type="xlm-roberta-base",
    lang="kk",
    rescale_with_baseline=False,
    device="cuda",
    batch_size=8
)

def bert_f1_batch(cands, ref):
    refs = [ref] * len(cands)
    _, _, F1 = scorer.score(cands, refs)
    return [float(x) for x in F1.detach().cpu().tolist()]

# -----------------------
# 4) build DPO pairs
# -----------------------
rows = []
for idx, (prompt, ref) in enumerate(pairs, 1):
    cands = [generate_one(prompt) for _ in range(K_CANDIDATES)]
    scores = bert_f1_batch(cands, ref)

    best_i = max(range(K_CANDIDATES), key=lambda i: scores[i])
    worst_i = min(range(K_CANDIDATES), key=lambda i: scores[i])

    rows.append({
        "prompt": prompt,
        "chosen": cands[best_i],
        "rejected": cands[worst_i],
        "chosen_score": scores[best_i],
        "rejected_score": scores[worst_i],
    })

    if idx % 10 == 0:
        print(f"{idx}/{len(pairs)}")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", OUT_PATH)

# -----------------------
# 5) sanity-check lengths
# -----------------------
def tok_len(s: str) -> int:
    return len(tok(s, add_special_tokens=False)["input_ids"])

lens = [tok_len(r["chosen"]) for r in rows]
print("chosen tok min/avg/max:", min(lens), sum(lens)//len(lens), max(lens))


## DPO training

In [ ]:
# Install + imports + memory settings
!pip -q install -U trl transformers datasets peft accelerate bitsandbytes

import os, gc, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
# Paths + load dataset + split
BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685_final_project"

MODEL_BASE = "issai/LLama-3.1-KazLLM-1.0-8B"

SFT_REALISER_DIR = os.path.join(BASE_PATH, "models/realizer_kazllm")
DPO_DATA_PATH = os.path.join(BASE_PATH, "data/dpo_pairs_realiser_train_100_v3_gpu.jsonl")
DPO_OUT_DIR = os.path.join(BASE_PATH, "models/realizer_dpo_kazllm_v3")

ds = load_dataset("json", data_files=DPO_DATA_PATH, split="train").shuffle(seed=42)
splits = ds.train_test_split(test_size=0.1, seed=42)
train_ds = splits["train"]
eval_ds  = splits["test"]

print("train:", len(train_ds), "eval:", len(eval_ds))
print("columns:", train_ds.column_names)  # must be prompt/chosen/rejected


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, token=True, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [ ]:
# Load base (4-bit) + attach adapters (dpo_train + dpo_ref)

import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=True,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    max_memory={0: "21GiB", "cpu": "64GiB"},
)

base.config.use_cache = False
base.config.pad_token_id = tokenizer.eos_token_id

TRAIN_ADAPTER = "dpo_train"
REF_ADAPTER   = "dpo_ref"

model = PeftModel.from_pretrained(
    base,
    SFT_REALISER_DIR,
    is_trainable=True,
    adapter_name=TRAIN_ADAPTER,
)

# load same weights as frozen reference adapter
model.load_adapter(SFT_REALISER_DIR, adapter_name=REF_ADAPTER)
model.set_adapter(TRAIN_ADAPTER)

print("Active adapter:", model.active_adapter)
print("Adapters:", list(model.peft_config.keys()))


In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_args = DPOConfig(
    output_dir=DPO_OUT_DIR,
    bf16=True,
    gradient_checkpointing=True,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=5e-6,
    num_train_epochs=1,

    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # prompt = CTRL+PLAN (long); keep room for story tokens
    max_prompt_length=1024,   # if OOM -> 768
    max_length=2048,          # if OOM -> 1536
    truncation_mode="keep_start",

    model_adapter_name="dpo_train",
    ref_adapter_name="dpo_ref",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,              # ref taken from ref_adapter_name
    args=dpo_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print("Trainer ready.")


In [ ]:
trainer.train()

trainer.save_model(DPO_OUT_DIR)
tokenizer.save_pretrained(DPO_OUT_DIR)

print("Saved DPO adapter to:", DPO_OUT_DIR)


# DPO chosen is from manually checked stories and rejected is from the worst score from the realizer model

## Preparing the dataset

In [ ]:
!pip -q install -U datasets transformers peft bitsandbytes accelerate bert-score trl

import os, json, re, random, gc
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from bert_score import score as bertscore

from google.colab import drive
drive.mount("/content/drive")

BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685_final_project"

MANUAL_JSON = "/mnt/data/manually_checked_plan.json"
MODEL_BASE = "issai/LLama-3.1-KazLLM-1.0-8B"


REALIZER_SFT_DIR = os.path.join(BASE_PATH, "models/realizer_kazllm")


DPO_PAIRS_OUT = os.path.join(BASE_PATH, "data/dpo_pairs_from_manual_45.jsonl")

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
MANUAL_JSON = "/content/manually_checked_plan.json"
def serialize_plan(planner_json: dict) -> str:
    chars = planner_json.get("characters", [])

    if isinstance(chars, list) and len(chars) > 0 and isinstance(chars[0], dict):
        char_str = ", ".join([f"{c.get('name','')} ({c.get('role','')})".strip() for c in chars])
    else:
        char_str = ", ".join([str(x) for x in chars])

    p = planner_json.get("plot", {})
    dev = p.get("development", [])
    dev_str = " ".join(dev) if isinstance(dev, list) else str(dev)

    return (
        f"[PLAN_START]\n"
        f"[SETTING] {planner_json.get('setting', 'Unknown')}\n"
        f"[CHARACTERS] {char_str}\n"
        f"[PLOT_BEGINNING] {p.get('beginning', '')}\n"
        f"[PLOT_CONFLICT] {p.get('conflict', '')}\n"
        f"[PLOT_DEVELOPMENT] {dev_str}\n"
        f"[PLOT_CLIMAX] {p.get('climax', '')}\n"
        f"[PLOT_RESOLUTION] {p.get('resolution', '')}\n"
        f"[MORAL] {p.get('moral', '')}\n"
        f"[PLAN_END]"
    )

def infer_length_tag(story: str) -> str:
    n = len(story.split())
    if n <= 120: return "very_short"
    if n <= 200: return "short"
    if n <= 350: return "medium"
    if n <= 600: return "long"
    return "very_long"

def extract_seed(story: str, max_chars: int = 120) -> str:
    s = story.strip().replace("\n", " ")
    m = re.split(r"(?<=[\.\!\?])\s+", s)
    seed = m[0] if m and len(m[0]) > 0 else s[:max_chars]
    return seed[:max_chars]

INST = "[INST] Write the fairy tale based on this plan.\n"

with open(MANUAL_JSON, "r", encoding="utf-8") as f:
    manual = json.load(f)

print("Loaded:", len(manual))

prompt_chosen = []
for ex in manual:
    age = ex.get("age_group", "unknown")
    theme = ex.get("theme") or ex.get("themes") or "unknown"
    if isinstance(theme, list):
        theme = ", ".join(theme)

    story = ex.get("story", "").strip()
    plan  = ex.get("planner", {})

    length_tag = infer_length_tag(story)
    seed_text  = extract_seed(story)

    tags = f"<AGE={age}> <THEME={theme}> <LENGTH={length_tag}> <SEED={seed_text}>"
    plan_str = serialize_plan(plan)

    prompt = f"[CTRL] {tags} [PLAN] {plan_str} {INST}"
    chosen = story

    if chosen:
        prompt_chosen.append({"prompt": prompt, "chosen": chosen})

print("prompt_chosen:", len(prompt_chosen))
print(prompt_chosen[0]["prompt"][:400])


In [ ]:
# --- load policy: base(4bit) + realizer SFT adapter ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, token=True, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=True,
    quantization_config=bnb,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
base.config.use_cache = False
base.config.pad_token_id = tokenizer.eos_token_id

policy = PeftModel.from_pretrained(base, REALIZER_SFT_DIR)
policy.eval()

@torch.no_grad()
def generate_one(prompt: str, max_new_tokens: int, temperature=0.8, top_p=0.9) -> str:
    x = tokenizer(prompt, return_tensors="pt").to(policy.device)
    y = policy.generate(
        **x,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )[0]
    full = tokenizer.decode(y, skip_special_tokens=True)

    return full[len(prompt):].strip() if full.startswith(prompt) else full.strip()

def bert_f1(cands, refs):
    P, R, F1 = bertscore(
        cands, refs,
        model_type="xlm-roberta-base",
        lang="kk",
        rescale_with_baseline=False,
        device=("cuda" if torch.cuda.is_available() else "cpu"),
        verbose=False,
        batch_size=8
    )
    return [float(x) for x in F1.cpu().tolist()]

K_CANDIDATES = 3
MAX_NEW_TOKENS = 512
rows = []
for i, ex in enumerate(prompt_chosen, 1):
    prompt = ex["prompt"]
    chosen = ex["chosen"]

    cands = [generate_one(prompt, max_new_tokens=MAX_NEW_TOKENS) for _ in range(K_CANDIDATES)]
    refs  = [chosen] * K_CANDIDATES
    scores = bert_f1(cands, refs)

    worst_i = min(range(K_CANDIDATES), key=lambda j: scores[j])
    rejected = cands[worst_i]

    rows.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

    if i % 5 == 0:
        print(f"{i}/{len(prompt_chosen)}")

with open(DPO_PAIRS_OUT, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:", DPO_PAIRS_OUT)


## Training DPO adapter

In [ ]:
# ===== ONE CELL: DPO (Realizer-only) full pipeline (load pairs -> split -> load 4bit base -> attach SFT adapters -> train -> optional eval -> save) =====
!pip -q install -U "trl>=0.12.0" datasets peft transformers bitsandbytes accelerate

import os, gc, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from trl import DPOTrainer, DPOConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- PATHS (yours) ----
BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685_final_project"

MODEL_BASE = "issai/LLama-3.1-KazLLM-1.0-8B"
REALIZER_SFT_DIR = os.path.join(BASE_PATH, "models/realizer_kazllm")
DPO_PAIRS_PATH   = os.path.join(BASE_PATH, "data/dpo_pairs_from_manual_45.jsonl")
DPO_OUT_DIR      = os.path.join(BASE_PATH, "models/realizer_dpo_from_manual_v1")

print("BASE_PATH:", BASE_PATH)
print("REALIZER_SFT_DIR:", REALIZER_SFT_DIR)
print("DPO_PAIRS_PATH:", DPO_PAIRS_PATH)
print("DPO_OUT_DIR:", DPO_OUT_DIR)

# ---- hard reset GPU memory (best effort) ----
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# ---- 1) Load DPO pairs + split ----
ds = load_dataset("json", data_files={"data": DPO_PAIRS_PATH})["data"]
assert set(["prompt","chosen","rejected"]).issubset(ds.column_names), ds.column_names
split = ds.train_test_split(test_size=0.2, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Pairs total/train/eval:", len(ds), len(train_ds), len(eval_ds))

# ---- 2) Tokenizer ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, token=True, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ---- 3) Auto-pick conservative lengths (main OOM lever) ----
def p95(lens):
    lens = sorted(lens)
    return lens[int(0.95 * (len(lens)-1))] if len(lens) > 1 else lens[0]

sample_n = min(40, len(train_ds))
prompt_lens, comp_lens = [], []
for i in range(sample_n):
    ex = train_ds[i]
    p = len(tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"])
    c = len(tokenizer(ex["chosen"], add_special_tokens=False)["input_ids"])
    r = len(tokenizer(ex["rejected"], add_special_tokens=False)["input_ids"])
    prompt_lens.append(p)
    comp_lens.append(max(c, r))

p95_prompt = p95(prompt_lens)
p95_comp   = p95(comp_lens)

MAX_PROMPT = int(min(384, p95_prompt))
MAX_LEN    = int(min(768, MAX_PROMPT + p95_comp + 16))
MAX_LEN    = max(MAX_LEN, MAX_PROMPT + 64)

print("p95_prompt:", p95_prompt, "p95_comp:", p95_comp)
print("Using MAX_PROMPT:", MAX_PROMPT, "MAX_LEN:", MAX_LEN)

# ---- 4) Load base 4-bit ----
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    token=True,
    quantization_config=bnb,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
base.config.use_cache = False
base.config.pad_token_id = tokenizer.eos_token_id

# memory savers
base.gradient_checkpointing_enable()
base.enable_input_require_grads()

# ---- 5) Attach two adapters INSIDE ONE MODEL (policy trainable + ref frozen) ----
# IMPORTANT: never use adapter_name="train" (conflicts with nn.Module.train())
POLICY_ADAPTER = "dpo_policy"
REF_ADAPTER    = "dpo_reference"

model = PeftModel.from_pretrained(
    base,
    REALIZER_SFT_DIR,
    is_trainable=True,
    adapter_name=POLICY_ADAPTER,
)
model.load_adapter(
    REALIZER_SFT_DIR,
    adapter_name=REF_ADAPTER,
    is_trainable=False
)
model.set_adapter(POLICY_ADAPTER)
model.train()

n_trainable = sum(p.requires_grad for p in model.parameters())
print("Adapters:", list(model.peft_config.keys()))
print("Active adapter:", model.active_adapter)
print("Params requiring grad:", n_trainable)
assert n_trainable > 0, "No trainable params (DPO won't learn)"

# ---- 6) Optional tiny eval subset (cheap) ----
EVAL_SUBSET_N = min(8, len(eval_ds))
eval_ds_small = eval_ds.select(range(EVAL_SUBSET_N)) if len(eval_ds) else None
print("Eval subset size:", (len(eval_ds_small) if eval_ds_small is not None else 0))

# ---- 7) DPO config (OOM-safe defaults) ----
cfg = DPOConfig(
    output_dir=DPO_OUT_DIR,
    report_to="none",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # if still OOM: set 2
    num_train_epochs=1,
    learning_rate=5e-6,
    bf16=True,

    logging_steps=1,

    # safest: no eval during training (saves memory)
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,

    max_prompt_length=MAX_PROMPT,
    max_length=MAX_LEN,
    beta=0.1,

    model_adapter_name=POLICY_ADAPTER,
    ref_adapter_name=REF_ADAPTER,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,                 # ref is inside model via REF_ADAPTER
    args=cfg,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

# ---- 8) Train ----
train_out = trainer.train()
print("TrainOutput:", train_out)

# ---- 9) Cheap eval metrics (Rewards/Logps/...) AFTER training ----
if eval_ds_small is not None and len(eval_ds_small) > 0:
    metrics = trainer.evaluate(eval_dataset=eval_ds_small)
    print("Eval metrics:", metrics)

# ---- 10) Save adapter ----
trainer.model.save_pretrained(DPO_OUT_DIR)
tokenizer.save_pretrained(DPO_OUT_DIR)
print("Saved DPO adapter to:", DPO_OUT_DIR)
